In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, recall_score, accuracy_score, precision_score, confusion_matrix,ConfusionMatrixDisplay



DATA_PATH = Path("cell2celltrain.csv")
SHEET_NAME = "cell2celltrain"


In [6]:
df = None

# TODO: Display the dimensions and first five rows.
# TECHNIQUE: Read the named worksheet from the Excel workbook and inspect a few records.
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Keep the Excel workbook beside this notebook."
    )

df = pd.read_csv(DATA_PATH)

print(f"Worksheet: {SHEET_NAME}")
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
display(df.head(5))

display(df.dtypes.rename("stored_type").to_frame())


Worksheet: cell2celltrain
Rows: 51047 | Columns: 58


,CustomerID,Churn,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls,PercChangeMinutes,PercChangeRevenues,...,ReferralsMadeBySubscriber,IncomeGroup,OwnsMotorcycle,AdjustmentsToCreditRating,HandsetPrice,MadeCallToRetentionTeam,CreditRating,PrizmCode,Occupation,MaritalStatus
0,3000002,1,24.00,219.0,22.0,0.25,0.0,0.0,-157.0,-19.0,...,0,4,No,0,30,Yes,1-Highest,Suburban,Professional,No
1,3000010,1,16.99,10.0,17.0,0.00,0.0,0.0,-4.0,0.0,...,0,5,No,0,30,No,4-Medium,Suburban,Professional,Yes
2,3000014,0,38.00,8.0,38.0,0.00,0.0,0.0,-2.0,0.0,...,0,6,No,0,Unknown,No,3-Good,Town,Crafts,Yes
3,3000022,0,82.28,1312.0,75.0,1.24,0.0,0.0,157.0,8.1,...,0,6,No,0,10,No,4-Medium,Other,Other,No
4,3000026,1,17.14,0.0,17.0,0.00,0.0,0.0,0.0,-0.2,...,0,9,No,1,10,No,1-Highest,Other,Professional,Yes


,stored_type
CustomerID,int64
Churn,int64
MonthlyRevenue,float64
MonthlyMinutes,float64
TotalRecurringCharge,float64
DirectorAssistedCalls,float64
OverageMinutes,float64
RoamingCalls,float64
PercChangeMinutes,float64
PercChangeRevenues,float64


In [29]:
TARGET = "Churn"
IDENTIFIER = "CustomerID"
STRING_FIELDS = ["ServiceArea" , "ChildrenInHH" ,
"HandsetRefurbished" ,
"HandsetWebCapable" ,
"TruckOwner" ,
"RVOwner" ,
"Homeownership" ,
"BuysViaMailOrder" ,
"RespondsToMailOffers" ,
"OptOutMailings" ,
"NonUSTravel" , "OwnsComputer" ,
"HasCreditCard" ,"NewCellphoneUser" ,
"NotNewCellphoneUser" ,
"OwnsMotorcycle" , "HandsetPrice" ,
"MadeCallToRetentionTeam" ,
"CreditRating" ,
"PrizmCode" ,
"Occupation" ,
"MaritalStatue"
]

EXCLUDED_FIELDS = [IDENTIFIER , STRING_FIELDS]

dropped = df.dropna()


X = dropped.drop(columns=[TARGET , IDENTIFIER, "ServiceArea" , "ChildrenInHH" ,
"HandsetRefurbished" ,
"HandsetWebCapable" ,
"TruckOwner" ,
"RVOwner" ,
"Homeownership" ,
"BuysViaMailOrder" ,
"RespondsToMailOffers" ,
"OptOutMailings" ,
"NonUSTravel" , "OwnsComputer" ,
"HasCreditCard" ,"NewCellphoneUser" ,
"NotNewCellphoneUser" ,
"OwnsMotorcycle" , "HandsetPrice" ,
"MadeCallToRetentionTeam" ,
"CreditRating" ,
"PrizmCode" ,
"Occupation" ,
"MaritalStatus"
])
y = dropped[TARGET]
# Keep the class proportions similar in the training and test sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,
                                                    stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]  # Probability of churn (class 1).
pred_50 = (proba >= 0.50).astype(int)
pred_30 = (proba >= 0.30).astype(int)
summary = {'samples': len(df), 'test_accuracy_50': round(float((pred_50 == y_test).mean()), 3),
           'roc_auc': round(float(roc_auc_score(y_test, proba)), 3),
           'recall_50': round(float(recall_score(y_test, pred_50,zero_division=0)), 3),
           'recall_30': round(float(recall_score(y_test, pred_30,zero_division=0)), 3)}
print('Results:', summary)

Results: {'samples': 51047, 'test_accuracy_50': 0.712, 'roc_auc': 0.609, 'recall_50': 0.02, 'recall_30': 0.496}
